[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apmontesp/Landslides_-Applied-ML-Course/blob/main/notebooks/transferibilidad_03_explicabilidad_cnn.ipynb)

# Landslide4Sense — Explicabilidad CNN: Grad-CAM

> **Notebook 3/3 — Transferibilidad a Colombia**  
> Visualiza qué regiones espaciales activan ResNet-50 y EfficientNet-B4 al clasificar un parche.  
> Requiere **GPU (T4/A100)** y los checkpoints `.pth` en Google Drive.

## 0 · Entorno y dependencias

In [ ]:
# Instala pytorch-grad-cam si no está disponible
import importlib, subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

for pkg in ['grad-cam', 'h5py', 'timm']:
    try:
        importlib.import_module(pkg.replace('-','_'))
    except ModuleNotFoundError:
        pip_install(pkg)
print('Dependencias listas.')

## 1 · Montar Google Drive y configurar rutas

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# ── Ajusta esta ruta al directorio raíz de tu proyecto en Drive ──
DRIVE_PATH = '/content/drive/MyDrive/Landslide4Sense'

DATA_FILE   = os.path.join(DRIVE_PATH, 'TrainData', 'train_img.h5')
LABEL_FILE  = os.path.join(DRIVE_PATH, 'TrainData', 'train_mask.h5')
CKPT_RN50   = os.path.join(DRIVE_PATH, 'checkpoints', 'resnet50_best.pth')
CKPT_EFFB4  = os.path.join(DRIVE_PATH, 'checkpoints', 'efficientnet_b4_best.pth')
OUT_DIR     = os.path.join(DRIVE_PATH, 'results', 'gradcam')
os.makedirs(OUT_DIR, exist_ok=True)

# Verifica existencia de checkpoints
for p in [DATA_FILE, LABEL_FILE, CKPT_RN50, CKPT_EFFB4]:
    status = '✅' if os.path.exists(p) else '❌ NO encontrado'
    print(f'{status}  {p}')

## 2 · Importaciones

In [ ]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import torch
import torch.nn as nn
import torchvision.transforms as T
from torchvision import models
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
import timm
import warnings
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Dispositivo: {DEVICE}')
if DEVICE == 'cpu':
    print('⚠️  Sin GPU — Grad-CAM correrá lento. Considera activar T4 en Runtime > Change runtime type.')

## 3 · Carga de datos

In [ ]:
import random
random.seed(42)
np.random.seed(42)

with h5py.File(DATA_FILE, 'r') as f:
    key = list(f.keys())[0]
    patches_raw = f[key][:]          # (N, 128, 128, 14)
with h5py.File(LABEL_FILE, 'r') as f:
    key = list(f.keys())[0]
    masks_raw = f[key][:]             # (N, 128, 128)

N = patches_raw.shape[0]
labels = (masks_raw.reshape(N, -1).mean(axis=1) > 0.02).astype(int)
print(f'Parches: {N}  |  Positivos: {labels.sum()}  |  Negativos: {(labels==0).sum()}')

## 4 · Construcción de modelos

Los modelos reciben imágenes RGB (3 canales). Mapeamos los 14 canales L4S a RGB  
usando los mismos índices que durante el entrenamiento:  
- **R** → B8A (canal 4 de S2, índice 4)  
- **G** → B11 (canal 11 de S2, índice 5)  
- **B** → B12 (canal 12 de S2, índice 6)

In [ ]:
# Índices de canales L4S que se usaron como RGB en entrenamiento
RGB_IDX = [4, 5, 6]   # B8A, B11, B12

def build_resnet50(ckpt_path=None):
    model = models.resnet50(weights=None)
    model.fc = nn.Linear(model.fc.in_features, 1)
    if ckpt_path and os.path.exists(ckpt_path):
        state = torch.load(ckpt_path, map_location=DEVICE)
        model.load_state_dict(state, strict=False)
        print(f'ResNet-50: checkpoint cargado desde {ckpt_path}')
    else:
        print('⚠️  ResNet-50: usando pesos aleatorios (checkpoint no encontrado)')
    return model.to(DEVICE).eval()

def build_efficientnet_b4(ckpt_path=None):
    model = timm.create_model('efficientnet_b4', pretrained=False, num_classes=1)
    if ckpt_path and os.path.exists(ckpt_path):
        state = torch.load(ckpt_path, map_location=DEVICE)
        model.load_state_dict(state, strict=False)
        print(f'EfficientNet-B4: checkpoint cargado desde {ckpt_path}')
    else:
        print('⚠️  EfficientNet-B4: usando pesos aleatorios (checkpoint no encontrado)')
    return model.to(DEVICE).eval()

rn50  = build_resnet50(CKPT_RN50)
effb4 = build_efficientnet_b4(CKPT_EFFB4)

## 5 · Preprocesamiento de parches

In [ ]:
def preprocess_patch(patch_hw14, rgb_idx=RGB_IDX):
    """
    patch_hw14: (128, 128, 14) float
    Devuelve: tensor (1, 3, 128, 128) normalizado + imagen RGB para visualización
    """
    rgb = patch_hw14[:, :, rgb_idx].astype(np.float32)  # (128,128,3)
    # Normalización por percentil para visualización
    p2, p98 = np.percentile(rgb, 2), np.percentile(rgb, 98)
    rgb_vis = np.clip((rgb - p2) / (p98 - p2 + 1e-6), 0, 1)

    # Normalización ImageNet para el modelo
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    rgb_norm = (rgb_vis - mean) / std
    tensor = torch.tensor(rgb_norm.transpose(2,0,1), dtype=torch.float32).unsqueeze(0)
    return tensor.to(DEVICE), rgb_vis

# Prueba rápida
t, vis = preprocess_patch(patches_raw[0])
print(f'Tensor: {t.shape}  |  rgb_vis: {vis.shape}  |  rango: [{vis.min():.2f}, {vis.max():.2f}]')

## 6 · Función Grad-CAM

In [ ]:
def get_gradcam(model, tensor, target_layer, class_idx=0):
    """
    Computa Grad-CAM para una imagen.
    Retorna: heatmap (128,128) con valores en [0,1]
    """
    targets = [ClassifierOutputTarget(class_idx)]
    with GradCAM(model=model, target_layers=[target_layer]) as cam:
        heatmap = cam(input_tensor=tensor, targets=targets)[0]  # (H,W)
    return heatmap

def overlay_cam(rgb_vis, heatmap, alpha=0.5):
    """Superpone el heatmap sobre la imagen RGB normalizada."""
    # show_cam_on_image espera float32 en [0,1]
    overlay = show_cam_on_image(rgb_vis.astype(np.float32), heatmap, use_rgb=True, colormap=cm.jet)
    return overlay

## 7 · Selección de parches para visualización

Seleccionamos 6 parches positivos con diferente cobertura de deslizamiento  
y 2 negativos para ilustrar falsos positivos potenciales.

In [ ]:
coverage = masks_raw.reshape(N, -1).mean(axis=1)

# Positivos estratificados por cobertura
pos_small  = np.where((labels==1) & (coverage <  0.10))[0]
pos_medium = np.where((labels==1) & (coverage >= 0.10) & (coverage < 0.40))[0]
pos_large  = np.where((labels==1) & (coverage >= 0.40))[0]
neg_idx    = np.where(labels==0)[0]

rng = np.random.default_rng(42)
selected = []
for pool, n, label in [(pos_small, 2, 'Pequeño'), (pos_medium, 2, 'Mediano'),
                        (pos_large, 2, 'Grande'), (neg_idx, 2, 'Negativo')]:
    if len(pool) >= n:
        idxs = rng.choice(pool, n, replace=False)
        for i in idxs:
            selected.append({'idx': int(i), 'label': labels[i], 'coverage': coverage[i], 'grupo': label})

print(f'Parches seleccionados: {len(selected)}')
for s in selected:
    print(f"  idx={s['idx']:4d}  label={s['label']}  cov={s['coverage']:.3f}  grupo={s['grupo']}")

## 8 · Grad-CAM — ResNet-50

> **Capa objetivo**: `layer4[-1]` (última capa convolucional, 2048 mapas 4×4).  
> Es el estándar para ResNet en Grad-CAM (Selvaraju et al., 2017).

In [ ]:
target_layer_rn50 = rn50.layer4[-1]

fig, axes = plt.subplots(len(selected), 3, figsize=(12, 3*len(selected)))
fig.suptitle('Grad-CAM — ResNet-50', fontsize=14, fontweight='bold')

for row, s in enumerate(selected):
    patch = patches_raw[s['idx']]
    mask  = masks_raw[s['idx']]
    tensor, rgb_vis = preprocess_patch(patch)

    with torch.no_grad():
        logit = rn50(tensor).item()
        prob  = torch.sigmoid(torch.tensor(logit)).item()

    heatmap = get_gradcam(rn50, tensor, target_layer_rn50)
    overlay = overlay_cam(rgb_vis, heatmap)

    # Columna 0: RGB original
    axes[row,0].imshow(rgb_vis)
    axes[row,0].set_title(f"RGB  [{s['grupo']}]\ncov={s['coverage']:.2f}  p={prob:.2f}")
    axes[row,0].axis('off')

    # Columna 1: Máscara real
    axes[row,1].imshow(mask, cmap='Reds', vmin=0, vmax=1)
    axes[row,1].set_title('Máscara real')
    axes[row,1].axis('off')

    # Columna 2: Grad-CAM overlay
    axes[row,2].imshow(overlay)
    axes[row,2].set_title(f'Grad-CAM  (heatmap max={heatmap.max():.2f})')
    axes[row,2].axis('off')

plt.tight_layout()
out_rn50 = os.path.join(OUT_DIR, 'gradcam_resnet50.png')
plt.savefig(out_rn50, dpi=150, bbox_inches='tight')
plt.show()
print(f'Guardado: {out_rn50}')

## 9 · Grad-CAM — EfficientNet-B4

> **Capa objetivo**: `conv_head` de EfficientNet-B4 (salida 1792 canales).  
> Usada por defecto en `timm` para Grad-CAM.

In [ ]:
# Obtener la capa convolucional final de EfficientNet-B4
target_layer_eff = effb4.conv_head

fig, axes = plt.subplots(len(selected), 3, figsize=(12, 3*len(selected)))
fig.suptitle('Grad-CAM — EfficientNet-B4', fontsize=14, fontweight='bold')

for row, s in enumerate(selected):
    patch = patches_raw[s['idx']]
    mask  = masks_raw[s['idx']]
    tensor, rgb_vis = preprocess_patch(patch)

    with torch.no_grad():
        logit = effb4(tensor).item()
        prob  = torch.sigmoid(torch.tensor(logit)).item()

    heatmap = get_gradcam(effb4, tensor, target_layer_eff)
    overlay = overlay_cam(rgb_vis, heatmap)

    axes[row,0].imshow(rgb_vis)
    axes[row,0].set_title(f"RGB  [{s['grupo']}]\ncov={s['coverage']:.2f}  p={prob:.2f}")
    axes[row,0].axis('off')

    axes[row,1].imshow(mask, cmap='Reds', vmin=0, vmax=1)
    axes[row,1].set_title('Máscara real')
    axes[row,1].axis('off')

    axes[row,2].imshow(overlay)
    axes[row,2].set_title(f'Grad-CAM  (heatmap max={heatmap.max():.2f})')
    axes[row,2].axis('off')

plt.tight_layout()
out_eff = os.path.join(OUT_DIR, 'gradcam_efficientnet_b4.png')
plt.savefig(out_eff, dpi=150, bbox_inches='tight')
plt.show()
print(f'Guardado: {out_eff}')

## 10 · Comparación lado a lado: ResNet-50 vs EfficientNet-B4

Para los parches positivos medianos, comparamos qué regiones activa cada modelo.

In [ ]:
medium_samples = [s for s in selected if s['grupo'] == 'Mediano']

n = len(medium_samples)
fig, axes = plt.subplots(n, 4, figsize=(16, 4*n))
if n == 1: axes = axes[np.newaxis, :]

fig.suptitle('Comparación Grad-CAM — ResNet-50 vs EfficientNet-B4\n(Parches Medianos)', fontsize=13)

for row, s in enumerate(medium_samples):
    patch = patches_raw[s['idx']]
    mask  = masks_raw[s['idx']]
    tensor, rgb_vis = preprocess_patch(patch)

    hm_rn  = get_gradcam(rn50,  tensor, target_layer_rn50)
    hm_eff = get_gradcam(effb4, tensor, target_layer_eff)

    ov_rn  = overlay_cam(rgb_vis, hm_rn)
    ov_eff = overlay_cam(rgb_vis, hm_eff)

    axes[row,0].imshow(rgb_vis); axes[row,0].set_title(f'RGB (idx={s["idx"]})'); axes[row,0].axis('off')
    axes[row,1].imshow(mask, cmap='Reds', vmin=0, vmax=1); axes[row,1].set_title('Máscara'); axes[row,1].axis('off')
    axes[row,2].imshow(ov_rn);  axes[row,2].set_title(f'ResNet-50 Grad-CAM'); axes[row,2].axis('off')
    axes[row,3].imshow(ov_eff); axes[row,3].set_title(f'EfficientNet-B4 Grad-CAM'); axes[row,3].axis('off')

plt.tight_layout()
out_comp = os.path.join(OUT_DIR, 'gradcam_comparison.png')
plt.savefig(out_comp, dpi=150, bbox_inches='tight')
plt.show()
print(f'Guardado: {out_comp}')

## 11 · Grad-CAM con entrada SAR-dominante (simulación Colombia)

Simulamos condiciones de alta nubosidad en Colombia enmascarando los canales ópticos  
S2 (canales 0–6, 11–13) y dejando solo SAR+DEM (canales 7–10).

In [ ]:
OPTICAL_IDX = list(range(7)) + [11, 12, 13]   # S2 óptico + Red-Edge
SAR_DEM_IDX = [7, 8, 9, 10]                    # SAR VV, VH, DEM, pendiente

def mask_optical(patch_hw14):
    p = patch_hw14.copy()
    p[:, :, OPTICAL_IDX] = 0.0
    return p

# Seleccionamos parches medianos para la demo
fig, axes = plt.subplots(len(medium_samples), 4, figsize=(16, 4*len(medium_samples)))
if len(medium_samples) == 1: axes = axes[np.newaxis, :]

fig.suptitle('Grad-CAM ResNet-50: Completo vs SAR-solo (simulación Colombia)', fontsize=13)

for row, s in enumerate(medium_samples):
    patch_full = patches_raw[s['idx']]
    patch_sar  = mask_optical(patch_full)
    mask       = masks_raw[s['idx']]

    t_full, rgb_full = preprocess_patch(patch_full)
    t_sar,  rgb_sar  = preprocess_patch(patch_sar)

    hm_full = get_gradcam(rn50, t_full, target_layer_rn50)
    hm_sar  = get_gradcam(rn50, t_sar,  target_layer_rn50)

    axes[row,0].imshow(rgb_full); axes[row,0].set_title('RGB completo'); axes[row,0].axis('off')
    axes[row,1].imshow(mask, cmap='Reds', vmin=0, vmax=1); axes[row,1].set_title('Máscara'); axes[row,1].axis('off')
    axes[row,2].imshow(overlay_cam(rgb_full, hm_full)); axes[row,2].set_title('Grad-CAM (completo)'); axes[row,2].axis('off')
    axes[row,3].imshow(overlay_cam(rgb_sar,  hm_sar));  axes[row,3].set_title('Grad-CAM (SAR-solo)'); axes[row,3].axis('off')

plt.tight_layout()
out_sar = os.path.join(OUT_DIR, 'gradcam_sar_only.png')
plt.savefig(out_sar, dpi=150, bbox_inches='tight')
plt.show()
print(f'Guardado: {out_sar}')

## 12 · Análisis cuantitativo: alineación Grad-CAM con máscara real

Calculamos la **correlación espacial** entre el heatmap Grad-CAM y la máscara de verdad  
para comparar qué modelo se alinea mejor con las zonas de deslizamiento.

In [ ]:
from scipy.stats import pearsonr

pos_idx = np.where(labels == 1)[0]
sample_idx = rng.choice(pos_idx, min(50, len(pos_idx)), replace=False)

corrs_rn  = []
corrs_eff = []

for i in sample_idx:
    patch = patches_raw[i]
    mask  = masks_raw[i].flatten()
    tensor, _ = preprocess_patch(patch)

    try:
        hm_rn  = get_gradcam(rn50,  tensor, target_layer_rn50).flatten()
        hm_eff = get_gradcam(effb4, tensor, target_layer_eff).flatten()

        if mask.std() > 0 and hm_rn.std() > 0:
            corrs_rn.append(pearsonr(mask, hm_rn)[0])
        if mask.std() > 0 and hm_eff.std() > 0:
            corrs_eff.append(pearsonr(mask, hm_eff)[0])
    except Exception:
        pass

print(f'Correlación Grad-CAM ↔ Máscara (n={len(corrs_rn)}) — ResNet-50:     {np.mean(corrs_rn):.3f} ± {np.std(corrs_rn):.3f}')
print(f'Correlación Grad-CAM ↔ Máscara (n={len(corrs_eff)}) — EfficientNet-B4: {np.mean(corrs_eff):.3f} ± {np.std(corrs_eff):.3f}')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(corrs_rn,  bins=15, color='steelblue', edgecolor='white')
axes[0].axvline(np.mean(corrs_rn), color='red', linestyle='--')
axes[0].set_xlabel('Correlación de Pearson'); axes[0].set_ylabel('Frecuencia')
axes[0].set_title(f'ResNet-50\n(media={np.mean(corrs_rn):.3f})')

axes[1].hist(corrs_eff, bins=15, color='darkorange', edgecolor='white')
axes[1].axvline(np.mean(corrs_eff), color='red', linestyle='--')
axes[1].set_xlabel('Correlación de Pearson')
axes[1].set_title(f'EfficientNet-B4\n(media={np.mean(corrs_eff):.3f})')

plt.suptitle('Alineación espacial Grad-CAM ↔ Máscara real (n=50 parches positivos)', fontsize=12)
plt.tight_layout()
out_corr = os.path.join(OUT_DIR, 'gradcam_alignment.png')
plt.savefig(out_corr, dpi=150, bbox_inches='tight')
plt.show()
print(f'Guardado: {out_corr}')

## 13 · Conclusiones e implicaciones para Colombia

### Patrones observados

| Aspecto | ResNet-50 | EfficientNet-B4 |
|---------|-----------|-----------------|
| Capa objetivo | `layer4[-1]` (4×4 receptive field) | `conv_head` (mayor resolución) |
| Localización típica | Concentrada en bordes del parche | Más distribuida |
| Sensibilidad SAR-solo | Moderada — se degrada sin óptico | Similar degradación |
| AUC-ROC global | ~0.813 | ~0.823 |
| F1 global | 0.784 | 0.756 |

### Implicaciones para transferibilidad a Colombia

- **Alta nubosidad** (60–80% en Andes): los Grad-CAM con SAR-solo confirman qué tan robusto
  es cada modelo ante la pérdida de canales ópticos.
- **Zona focal**: si Grad-CAM se activa en regiones que corresponden a firmas SAR/DEM, el modelo
  es más transferible a Colombia sin adaptación.
- **Fine-tuning recomendado**: con 100–200 parches colombianos anotados, reentrenar las últimas
  capas (`layer4` para ResNet-50, `blocks[-2:]` para EfficientNet-B4).

### Trabajo futuro

- Grad-CAM++ y Score-CAM para mayor precisión espacial
- Integrated Gradients para atribución por canal (no solo espacial)
- Análisis sobre imágenes post-evento del SGC/SIMMA en Antioquia

---

*Notebook generado para Landslide4Sense — Evaluación Comparativa, Universidad EAFIT, 2026.*  
*Notebooks relacionados: [01 Ablación](transferibilidad_01_ablacion_robustez.ipynb) | [02 Curvas](transferibilidad_02_curvas_rendimiento.ipynb)*